In [1]:
%load_ext autoreload
%autoreload 2 

In [3]:
from pathlib import Path
import os
import socket

In [4]:
try:
    project_root = next(p for p in Path().cwd().resolve().parents if (p /"env.env").exists())
except StopIteration:
    # Fallback to current working directory if not found
    project_root = Path.cwd()
print(f"Project Root: {project_root}")
ROOT_DATASET_DIR = project_root / "datasets" 

Project Root: /storage/ice1/1/0/vchopra37/projects/learning_env


In [5]:
# Check if running on the Pace server
hostname = socket.gethostname()
IS_PACE_SERVER = "pace" in hostname.lower()  # Replace 'pace' with a unique keyword from your server's hostname
IS_PACE_SERVER

True

In [6]:
import torch
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm

In [7]:
_ = torch.manual_seed(0)

In [8]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv( Path.cwd() / "env.env")
PATH_TO_DATA_DIR = Path.cwd() / "dataset"
PATH_TO_DATA_DIR

PosixPath('/storage/ice1/1/0/vchopra37/projects/learning_env/dataset')

### Step 1: Load pretrained GPT-2

In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [10]:
model_id = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)
print(model.config.to_dict())

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

{'transformers_version': '5.17.0', 'architectures': ['GPT2LMHeadModel'], 'output_hidden_states': False, 'return_dict': True, 'dtype': 'float32', 'chunk_size_feed_forward': 0, 'is_encoder_decoder': False, 'id2label': {0: 'LABEL_0', 1: 'LABEL_1'}, 'label2id': {'LABEL_0': 0, 'LABEL_1': 1}, 'problem_type': None, 'vocab_size': 50257, 'n_positions': 1024, 'n_embd': 768, 'n_layer': 12, 'n_head': 12, 'n_inner': None, 'activation_function': 'gelu_new', 'resid_pdrop': 0.1, 'embd_pdrop': 0.1, 'attn_pdrop': 0.1, 'layer_norm_epsilon': 1e-05, 'initializer_range': 0.02, 'summary_type': 'cls_index', 'summary_use_proj': True, 'summary_activation': None, 'summary_proj_to_labels': True, 'summary_first_dropout': 0.1, 'scale_attn_weights': True, 'use_cache': True, 'bos_token_id': 50256, 'eos_token_id': 50256, 'pad_token_id': None, 'scale_attn_by_inverse_layer_idx': False, 'reorder_and_upcast_attn': False, 'add_cross_attention': False, 'tie_word_embeddings': True, '_name_or_path': 'gpt2', 'model_type': 'gpt

In [11]:
print(f"Total parameters: {sum(parameter.numel() for parameter in model.parameters()):,}")

Total parameters: 124,439,808


In [12]:
sample = "Hello, who are you ? "
out = model.generate(tokenizer.encode(sample, return_tensors="pt"), max_new_tokens = 20)
print(tokenizer.decode(out[0], skip_special_tokens=True))

Hello, who are you ?  I'm a little bit of a nerd, but I'm not a big fan of the idea


#### Step 2 — Inspect GPT-2 Architecture

In [13]:
total_params = 0

for name, parameter in model.named_parameters():
    parameter_count = parameter.numel()
    total_params += parameter_count

    print(
        f"{name:<60} "
        f"shape={tuple(parameter.shape)!s:<20} "
        f"parameters={parameter_count:,} "
        f"trainable={parameter.requires_grad}"
    )

print(f"\nTotal parameters: {total_params:,}")

transformer.wte.weight                                       shape=(50257, 768)         parameters=38,597,376 trainable=True
transformer.wpe.weight                                       shape=(1024, 768)          parameters=786,432 trainable=True
transformer.h.0.ln_1.weight                                  shape=(768,)               parameters=768 trainable=True
transformer.h.0.ln_1.bias                                    shape=(768,)               parameters=768 trainable=True
transformer.h.0.attn.c_attn.weight                           shape=(768, 2304)          parameters=1,769,472 trainable=True
transformer.h.0.attn.c_attn.bias                             shape=(2304,)              parameters=2,304 trainable=True
transformer.h.0.attn.c_proj.weight                           shape=(768, 768)           parameters=589,824 trainable=True
transformer.h.0.attn.c_proj.bias                             shape=(768,)               parameters=768 trainable=True
transformer.h.0.ln_2.weight      

In [14]:
for name, module in model.named_modules():
    # print(module.__dict__.keys())
    print(f"{name:<50} {type(module).__name__}  " )

                                                   GPT2LMHeadModel  
transformer                                        GPT2Model  
transformer.wte                                    Embedding  
transformer.wpe                                    Embedding  
transformer.drop                                   Dropout  
transformer.h                                      ModuleList  
transformer.h.0                                    GPT2Block  
transformer.h.0.ln_1                               LayerNorm  
transformer.h.0.attn                               GPT2Attention  
transformer.h.0.attn.c_attn                        Conv1D  
transformer.h.0.attn.c_proj                        Conv1D  
transformer.h.0.attn.attn_dropout                  Dropout  
transformer.h.0.attn.resid_dropout                 Dropout  
transformer.h.0.ln_2                               LayerNorm  
transformer.h.0.mlp                                GPT2MLP  
transformer.h.0.mlp.c_fc                           Conv1D 

### Step 3 — Load and Inspect the Instruction Dataset

In [15]:
if IS_PACE_SERVER:
    from model_training.post_training.artifacts.data.instruction_tuning.raw_data.simple_example.instruct_data import download_and_load_file
    from model_training.post_training.artifacts.data.instruction_tuning.prompt_style.styles import AlpacaFormat
    from model_training.post_training.artifacts.data.instruction_tuning.dataloader import InstructionDataset, custom_collate_fn
else:
    ## Fix this import
    raise Exception("Not Running on Pace")
        # from src.model import GPTdataset['train']

In [16]:
from datasets import load_dataset, DatasetDict

data = load_dataset("yahma/alpaca-cleaned", split="train")
print(data)

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 51760
})


In [17]:
alpaca = AlpacaFormat(data)
## Example of format
print(alpaca.format_input(instruction=data[50]['instruction'], in_text=data[50]['input']))
print("=============================================")
print(alpaca.detailed_response(out_text = data[50]['output']))

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Generate a poem with 10 lines.


### Response:
A meadow filled with flowers, colorful and bright,
A sky of endless blue in the warm sunlight,
Birds singing melodies so pure and so true,
Nature's beauty bursting with every hue.

The rustling of leaves in a gentle breeze,
The buzzing of bees, busy as you please,
A river flowing with grace and with ease,
All coming together in perfect harmony.

The wonder of creation so vast and so grand,
Nature's poetry in the palm of your hand.


In [18]:
sample = (alpaca.complete_response(instruction=data[999]['instruction'], in_text=data[999]['input'], out_text=data[999]['output']))
print(sample)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Evaluate the feasibility and potential pitfalls of the following business idea: opening a high-end, gourmet donut shop in a small town with a population of only 5,000 people.

### Response:
The feasibility of opening a high-end gourmet donut shop in a small town of 5,000 people may be limited due to factors such as market size, demographics, and competition. A small population may not provide enough demand for such a specialized product like gourmet donuts, especially if there are no nearby towns or tourist attractions to draw additional customers. Furthermore, the demographics of the town may not align with the target market of a high-end gourmet donut shop, as residents in small towns may not necessarily have the disposable income or interest in gourmet donuts. In addition, potential competitors such as local bakeries or franchises may already have established c

#### Spliting the data 

In [19]:
# 1. Split the original dataset into 80% train and 20% remainder (test + valid)
train_remainder = data.train_test_split(test_size=0.2, seed=42)

# 2. Split the 20% remainder in half to get 10% test and 10% validation
test_valid = train_remainder["test"].train_test_split(test_size=0.5, seed=42)

# 3. Combine everything into a clean DatasetDict
final_dataset = DatasetDict({
    "train": train_remainder["train"],
    "validation": test_valid["train"],  # The 'train' side of the remainder split
    "test": test_valid["test"]          # The 'test' side of the remainder split
})

In [20]:
dataloader = InstructionDataset(ds=final_dataset['train'], tokenizer=tokenizer, prompt_style=alpaca)
(sample)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1098 > 1024). Running this sequence through the model will result in indexing errors


'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nEvaluate the feasibility and potential pitfalls of the following business idea: opening a high-end, gourmet donut shop in a small town with a population of only 5,000 people.\n\n### Response:\nThe feasibility of opening a high-end gourmet donut shop in a small town of 5,000 people may be limited due to factors such as market size, demographics, and competition. A small population may not provide enough demand for such a specialized product like gourmet donuts, especially if there are no nearby towns or tourist attractions to draw additional customers. Furthermore, the demographics of the town may not align with the target market of a high-end gourmet donut shop, as residents in small towns may not necessarily have the disposable income or interest in gourmet donuts. In addition, potential competitors such as local bakeries or franchises may already have establ